# Pin sets

This notebook demonstrates usage of workspace pin sets using `pyeodh`.

A pin set is a named list of pinned STAC items saved in a workspace. Items you pin in the Resource Catalogue web UI are saved to pin sets, so you can load them in a notebook, and sets you create here show up in the UI.

Note: This API requires authentication credentials to be provided by the user (in this case read from environment variables).

In [1]:
import os

import pyeodh

username = os.getenv("EODH_USERNAME")
token = os.getenv("EODH_TOKEN")

# Optionally specify a different server
client = pyeodh.Client(base_url="https://staging.eodatahub.org.uk", username=username, token=token)
ws = client.workspace

Let's pin a few Sentinel-2 scenes to a new set. Items can be any STAC items from the resource catalog.

In [2]:
catalog = client.get_catalog_service().get_catalog("public/catalogs/ceda-stac-catalogue")
scenes = list(catalog.search(collections=["sentinel2_ard"], bbox=[-1.2, 53.8, -0.9, 54.1], limit=3)[:3])

pin_set = ws.create_pin_set("York scenes", items=scenes)
print(pin_set.id, pin_set.name, pin_set.item_count)

470b0fd9-5d05-44c9-b566-0df7c3e3f235 York scenes 3


List the pin sets in your workspace, including the ones pinned in the web UI:

In [3]:
for summary in ws.list_pin_sets():
    print(summary.id, summary.name, summary.item_count)

470b0fd9-5d05-44c9-b566-0df7c3e3f235 York scenes 3


Get a set and load its items. Items you don't have access to are skipped with a warning.

In [4]:
pin_set = ws.get_pin_set(pin_set.id)

for item in pin_set.load_items():
    print(item.datetime, item.id)

2026-10-05 11:21:11+00:00 neodc.sentinel_ard.data.sentinel_2.2026.10.05.S2C_20261005_latn545lonw0006_T30UXF_ORB037_20261005115326_utm30n_osgb
2026-10-05 11:21:11+00:00 neodc.sentinel_ard.data.sentinel_2.2026.10.05.S2C_20261005_latn536lonw0007_T30UXE_ORB037_20261005115326_utm30n_osgb
2026-10-04 11:08:31+00:00 neodc.sentinel_ard.data.sentinel_2.2026.10.04.S2A_20261004_latn545lonw0006_T30UXF_ORB137_20261004162422_utm30n_osgb


Sets can also be renamed, edited and shared, see the [API documentation](https://pyeodh.readthedocs.io/en/latest/api.html#module-pyeodh.workspace). Finally, delete the example set:

In [5]:
ws.delete_pin_set(pin_set.id)